Gaia Field Stars (CMD)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from astropy.table import QTable, vstack
from astropy import units as u

from matplotlib.patches import Rectangle, Circle, Ellipse, Polygon

from astroquery.gaia import Gaia


Instructions

Enter the coordinates of the center of your field (
l
,
b
).

Use a galactic longitude (
l
) of somewhere between 10 and 90 degrees
Use a galactic latitude (
b
) of somewhere between +35 and +45 degrees

In [ ]:
my_object_l = 20
my_object_b = 40


Get the the Gaia data for your field

Create and run a Gaia query to find a
high quality complete representitive sample
of 5,000 +/- 200 objects that are brighter than 17th magnutude, have a parallax SNR > 15, and have the bp-rp colors.
Adjust the radius of your search circular search window and make sure you are not cutting off any objects. See the
Note on SELECT-ORDER Bias issues
section from the HQ_Gaia notebook.
You will want the columns:
source_id, l, b, phot_g_mean_mag, bp_rp, parallax, parallax_over_error
Don't print out the entire object table!

In [ ]:
my_query = f"""
SELECT TOP 5500
source_id, l, b, phot_g_mean_mag, bp_rp, parallax, parallax_over_error
FROM gaiadr3.gaia_source_lite
WHERE DISTANCE( POINT({my_object_l}, {my_object_b}), POINT(l, b) ) < 1.47
AND phot_g_mean_mag  < 17
AND parallax_over_error > 15
AND bp_rp IS NOT NULL
ORDER BY parallax DESC
"""


In [ ]:
my_job_query = Gaia.launch_job_async(my_query)


In [ ]:
print(my_job_query)


In [ ]:
my_table = my_job_query.get_results()
my_table[0:2]


How many objects did your query return?

In [ ]:
np.size(my_table)


Use
parallax
and
phot_g_mean_mag
to get
distance
(in pc) and the
absolute magnitude
($G_{M}$)

Add them as columns to the data table

In [ ]:
my_table['distance'] = my_table['parallax'].to(u.parsec, equivalencies=u.parallax())


In [ ]:
def find_absmag(my_gmag, my_distance):
    result = my_gmag - 5 * np.log10( my_distance / (10 * u.parsec)) * u.mag
    return result

my_table['abs_g'] = find_absmag(my_gmag = my_table['phot_g_mean_mag'], 
                                 my_distance = my_table['distance']
                               ) * u.mag


In [ ]:
my_table[0:2]


Make a subset of all of the
white dwarf
stars in your field

You will need information from the CMD on the back your
Gaia Info Sheet
.
You will need to make a cut based on the color (
bp_rp
) and Absolute magnitude (G$_M$).
It may be easier to make a few smaller tables and combine them.

In [ ]:
my_dwarf_table1 = my_table[(my_table['bp_rp'] > 1) &
    (my_table['bp_rp'] < 2) &
    (my_table['abs_g'] > 13)]
my_dwarf_table2 = my_table[(my_table['bp_rp'] < 1) &
    (my_table['abs_g'] > 12) ]
my_dwarf_table3 = my_table[(my_table['bp_rp'] < 0.7) &
    (my_table['abs_g'] < 12) &
                            (my_table['abs_g'] > 8)]


In [ ]:
my_dwarf_table = vstack(
    [my_dwarf_table1, my_dwarf_table2, my_dwarf_table3]
)


How many of your objects are white dwarf stars?

In [ ]:
np.size(my_dwarf_table)


Make a subset of the 300 nearest stars

In [ ]:
my_table.sort('distance')
my_near300=my_table[:300]


Make a subset of the 300 most distant stars

In [ ]:
my_table.sort('distance')
my_distant300=my_table[-300:]


Create the following plot

All on one page using
.subplot_mosaic
Total size 12 inches wide, 15 inches high
With the following relative proportions:
--------------------
          |                  |
          |                  |
          |                  |
          |                  |
          |      Plot A      |
          |                  |
          |                  |
          |                  |
          |                  |
          --------------------
          |                  |
          |     Plot B       |
          --------------------
Plot A - An annotated color magnitude diagram (
bp_rp
vs. $G_{M}$) of your field.

Both axes labeled
A title for the figure
Plot all of the stars
A symbol and label placed at the position of the Sun (
bp_rp
= 0.82, $G_{M}$ = 4.67)
White dwarf stars plotted with a different symbol, color, and size
300 nearest stars plotted with a different symbol, color, and size
300 most distant stars plotted with a different symbol, color, and size
Use annotations (arrows, shapes, text) to make everything clear
Plot B - A histrogram of the
bp_rp
colors of the stars in your field.

Three histograms on the same plot
Set the Y-axis to
log
Use a bin size of 0.1 mag
A histogram of
ALL
of the stars in the field
A histogram of the 300 nearest stars in the field (in a different color)
A histogram of the 300 most distant stars in the field (in a different color)
All three histograms should be clearly labeled and easy to distinguish
Both axes labeled

In [ ]:
fig, ax = plt.subplot_mosaic(
    '''
    AA
    AA
    BB
    ''',
    figsize = (12, 15), 
    constrained_layout = True
)

my_sun_bprp=0.82
my_sun_absg=4.67

bp_rp=my_table["bp_rp"]
my_bins = np.arange(bp_rp.min(), bp_rp.max() + 0.1, 0.1)

ax['A'].set_aspect(1/4)

ax['A'].set_ylim(-4,15)
ax['A'].invert_yaxis()

ax['A'].set_xlabel("BP - RP",
              fontfamily = 'serif',
              fontsize = 25)

ax['A'].set_ylabel("G_Mag",
              fontfamily = 'serif',
              fontsize = 25)

ax['A'].set_title("CMD (l=20, b=40)",
             fontfamily = 'serif',
             fontsize = 30)

ax['A'].plot(my_table['bp_rp'], my_table['abs_g'],
        color = "#95d0fc",
        marker = ".",
        linestyle = "None", #point들 사이의 연결 X
        markersize = 5
       );

ax['A'].plot(my_dwarf_table['bp_rp'], my_dwarf_table['abs_g'],
        color = "coral",
        marker = "D",
        linestyle = "None", #point들 사이의 연결 X
        markersize = 5
       );

ax['A'].plot(my_near300['bp_rp'], my_near300['abs_g'],
        color = "#b790d4",
        marker = "^",
        linestyle = "None", #point들 사이의 연결 X
        markersize = 5
       );

ax['A'].plot(my_distant300['bp_rp'], my_distant300['abs_g'],
        color = "#40a368",
        marker = "s",
        linestyle = "None", #point들 사이의 연결 X
        markersize = 5
       );

ax['A'].plot(my_sun_bprp, my_sun_absg,
        color = "k",
        marker = "*",
        linestyle = "None",
        markersize = 20
       );

ax['A'].annotate('Sun', 
             xy = (0.75, 4.67), 
             xytext = (0.4, 7.5), #text 위치
             fontsize = 12, #txt 크기
             color = 'k', #text 색상
             arrowprops = {'color' : 'k', #arrow properties 성질
                           'linewidth' : 1,
                           'arrowstyle' : '->, head_length = 0.3, head_width = 0.3',
                           'connectionstyle' : 'angle3, angleA = 90, angleB = 0'}
           );

ax['A'].annotate('Nearest 300', 
             xy = (2.3, 10), 
             xytext = (2, 13), #text 위치
             fontsize = 12, #txt 크기
             color = '#b790d4', #text 색상
             arrowprops = {'color' : '#b790d4', #arrow properties 성질
                           'linewidth' : 1,
                           'arrowstyle' : '->, head_length = 0.3, head_width = 0.3',
                           'connectionstyle' : 'angle3, angleA = 90, angleB = 0'}
           );

ax['A'].annotate('Most Distant 300', 
             xy = (1.6, 0), 
             xytext = (2, 3), #text 위치
             fontsize = 12, #txt 크기
             color = '#40a368', #text 색상
             arrowprops = {'color' : '#40a368', #arrow properties 성질
                           'linewidth' : 1,
                           'arrowstyle' : '->, head_length = 0.3, head_width = 0.3',
                           'connectionstyle' : 'angle3, angleA = 90, angleB = 0'}
           );

ax['A'].annotate('White Dwarf', 
             xy = (-0.1, 12), 
             xytext = (-0.4,14), #text 위치
             fontsize = 12, #txt 크기
             color = 'coral', #text 색상
             arrowprops = {'color' : 'coral', #arrow properties 성질
                           'linewidth' : 1,
                           'arrowstyle' : '->, head_length = 0.3, head_width = 0.3',
                           'connectionstyle' : 'angle3, angleA = 90, angleB = 0'}
           );


ax['B'].set_yscale("log")

ax['B'].set_xlabel("BP - RP",
              fontfamily = 'serif',
              fontsize = 25)

ax['B'].set_ylabel("Number of Stars",
              fontfamily = 'serif',
              fontsize = 25)

ax['B'].hist(my_near300['bp_rp'],
        bins = my_bins,
        histtype = 'stepfilled',
        facecolor = 'MediumOrchid',
        label = "Nearest 300",
        alpha=0.3)
ax['B'].hist(my_near300['bp_rp'],
        bins = my_bins,
        histtype = 'step',
        linewidth=2,
        color = 'MediumOrchid')

# Histogram of the values of column 2 (Exam2)

ax['B'].hist(my_distant300['bp_rp'],
        bins = my_bins,
        histtype = 'stepfilled',
        color = 'MidnightBlue',
        label = "Most Distant",
        alpha=0.3)
ax['B'].hist(my_distant300['bp_rp'],
        bins = my_bins,
        histtype = 'step',
        linewidth=2,
        color = 'MidnightBlue')

ax['B'].hist(my_table['bp_rp'],
        bins = my_bins,
        histtype = 'stepfilled',
        color = '#d8dcd6',
        label = "All",
        alpha=0.3 )
ax['B'].hist(my_table['bp_rp'],
        bins = my_bins,
        histtype = 'step',
        linewidth=2,
        color = '#d8dcd6')

ax['B'].legend(loc=0, shadow=True);


Save and download a PNG version of your figure. You will need this for the final assignment of the class.

Feel free to change the name of the file if you wish.

In [ ]:
fig.savefig('CMD_plot.png', bbox_inches='tight')


Due Mon Feb 9 - 1 pm

File -> Download as -> HTML (.html)
upload your .html file to the class Canvas page